# Infer.NET 07b (Python) — TrueSkill : formules fermées, du cours au service web

**Série** : Programmation Probabiliste avec Infer.NET (7b/19)  
**Durée estimée** : 30 minutes  
**Prérequis** : [Infer-07-TrueSkill](Infer-07-TrueSkill.ipynb) et son jumeau [Infer-07b-TrueSkill-Formules-Fermees-CSharp](Infer-07b-TrueSkill-Formules-Fermees-CSharp.ipynb)

***

## Navigation

| précédent | Suivant |
|-----------|--------|
| [Infer-07-TrueSkill](Infer-07-TrueSkill.ipynb) | [Infer-08-Classification](Infer-08-Classification.ipynb) |

**Jumeau Python de `Infer-07b-TrueSkill-Formules-Fermees-CSharp.ipynb`** et compagnon de `Infer-07-TrueSkill.ipynb`.

**Carnet voisin, autre régime d'inférence** : [PyMC-07-TrueSkill](../PyMC/PyMC-07-TrueSkill.ipynb) traite le **même modèle** de compétences par **échantillonnage MCMC** (NUTS, un `pm.Potential` par contrainte), là où ce carnet dérive les **formules fermées EP** — sa section 7 compare les deux régimes et sa section 7 bis introduit le noyau EP que ce carnet déroule intégralement.

Ce carnet porte les **formules fermées EP** de TrueSkill (Herbrich, Minka, Graepel — *TrueSkill(TM): A Bayesian Skill Rating System*, NIPS 19 / 2006) en Python pur (`math.erf`), puis les expose comme un **service web** via Jupyter Kernel Gateway, sur le modèle du précédent `Z3-02-Sudoku-Python` (Epic #19571, pli 0 v2).

Objectifs pédagogiques :
1. Comprendre la mise à jour EP closed-form d'un match (fonctions `V` et `W`, moment matching du marginal incliné).
2. Dérouler l'apprentissage **en ligne** (une passe séquentielle par partie) sur 1v1, équipes et free-for-all.
3. Traiter le **match nul** (troncature bilatérale, marge de nul).
4. Exposer le modèle comme un **endpoint REST** (`POST /probas/trueskill`), un seul fichier pour deux usages : le carnet qu'on lit et le service qu'on appelle.

L'organe natif du calcul est la **forme fermée EP**, implémentée ici en Python ; la bibliothèque publique `trueskill` (Heungsub Lee) sert **uniquement de vérification indépendante** dans les tests, jamais d'organe de calcul (règle *organ-first*).

In [1]:
# Imports
import json
import math
import os
import sys
from typing import Dict, List, Optional, Sequence, Tuple


## 1. Le modèle TrueSkill

Chaque joueur a une compétence inconnue `s`, modélisée par une Gaussienne `N(mu, sigma^2)`. Sa **performance** dans une partie est `s + bruit`, où le bruit suit `N(0, beta^2)`. On observe l'**issue** du match (victoire, défaite, nul), jamais la compétence elle-même : l'inférence consiste à mettre à jour la croyance `(mu, sigma)` de chaque joueur après chaque partie.

Paramètres par défaut (identiques à `Infer-07`) :
- `mu0 = 25`, `sigma0 = 25/3` (le prior est large : on sait peu de chose) ;
- `beta = sigma0 / 2` (le bruit de performance).

Le cas de référence de cette section : **A bat B**, tous deux au prior. Le posterior publié (`Infer-07` §3) est `mu_A ≈ 29.21`, `mu_B ≈ 20.79`, `sigma ≈ 7.19`.


In [2]:
# Parametres TrueSkill par defaut (identiques a Infer-07 / Infer-07b-CSharp).
MU0 = 25.0
SIGMA0 = 25.0 / 3.0
BETA = SIGMA0 / 2.0

print(f"Prior : mu={MU0}, sigma={SIGMA0:.4f}, beta={BETA:.4f}")


Prior : mu=25.0, sigma=8.3333, beta=4.1667


## 2. La mise à jour EP d'un match : fonctions `V` et `W`

La mise à jour d'Expectation-Propagation pour un match `w bat l` a une **forme fermée**. En notant `c = sqrt(var_w + var_l + 2*beta^2)` et `t = (mu_w - mu_l) / c`, on définit :

- `V(t) = phi(t) / Phi(t)` (fonction de troncature de Herbrich 2006) ;
- `W(t) = V(t) * (V(t) + t)`.

Alors le moment matching **exact** du marginal incliné (identité du probit) donne :

```
mu_w   += var_w / c * V(t)
var_w  *= 1 - var_w / c^2 * W(t)
mu_l   -= var_l / c * V(t)
var_l  *= 1 - var_l / c^2 * W(t)
```

Le gagnant monte, le perdant descend, et les deux variances **diminuent** : chaque partie apporte de l'information. Implémentons ces formules en Python pur.


In [3]:
def phi(t: float) -> float:
    """Densite de la Gaussienne centree reduite."""
    return math.exp(-0.5 * t * t) / math.sqrt(2.0 * math.pi)


def Phi(t: float) -> float:
    """Fonction de repartition de la Gaussienne centree reduite (via math.erf)."""
    return 0.5 * (1.0 + math.erf(t / math.sqrt(2.0)))


def V(t: float) -> float:
    """V(t) = phi(t) / Phi(t), troncature de Herbrich 2006."""
    return phi(t) / Phi(t)


def W(t: float) -> float:
    """W(t) = V(t) * (V(t) + t)."""
    v = V(t)
    return v * (v + t)


def match_update(winner, loser, beta2):
    """Mise a jour EP closed-form d'un match (w bat l).

    Entrees/sorties : couples (mu, variance). beta2 = variance de performance
    additionnelle portee par le facteur de comparaison (2*beta^2 pour un 1v1).
    """
    mw, vw = winner
    ml, vl = loser
    c = math.sqrt(vw + vl + beta2)
    t = (mw - ml) / c
    vt, wt = V(t), W(t)
    mu_w = mw + vw / c * vt
    var_w = vw * (1.0 - vw / (c * c) * wt)
    mu_l = ml - vl / c * vt
    var_l = vl * (1.0 - vl / (c * c) * wt)
    return (mu_w, var_w), (mu_l, var_l)


# Cas de reference : A bat B, tous deux au prior.
v0 = SIGMA0 ** 2
(mu_a, var_a), (mu_b, var_b) = match_update((MU0, v0), (MU0, v0), 2.0 * BETA ** 2)
print(f"A bat B (1v1, priors egaux) :")
print(f"  mu_A = {mu_a:.4f}  (attendu ~29.21)")
print(f"  mu_B = {mu_b:.4f}  (attendu ~20.79)")
print(f"  sigma = {math.sqrt(var_a):.4f}  (attendu ~7.19)")


A bat B (1v1, priors egaux) :
  mu_A = 29.2052  (attendu ~29.21)
  mu_B = 20.7948  (attendu ~20.79)
  sigma = 7.1945  (attendu ~7.19)


### Interprétation : les moments gaussiens d'Infer-07

On retrouve les valeurs publiées d'`Infer-07` §3 : **mu_A ≈ 29.21**, **mu_B ≈ 20.79**, **sigma ≈ 7.19**. La symétrie est conservée (`mu_A + mu_B = 2 * mu0 = 50`), et la variance a nettement baissé (`sigma` passe de `8.33` à `7.19`) : un seul match a déjà beaucoup appris.

C'est le même résultat que le moteur **Infer.NET** (qui itère l'EP) et que le jumeau **C#** (formes fermées). En 1v1, l'identité du probit donne **exactement les moments** de la mise à jour EP par appariement de moments ; elle ne donne pas la loi postérieure complète sous forme fermée.


## 3. Apprentissage en ligne : équipes et séquences

En production, TrueSkill traite les parties **en ligne** : une passe séquentielle, jamais itérée. Une **équipe** est réduite à la **somme** des compétences de ses joueurs :

- `mu_equipe = somme des mu_joueurs` ;
- `var_equipe = somme des var_joueurs` ;
- le bruit de performance ajoute `beta^2` **par joueur** des deux équipes.

Après la mise à jour d'équipe (`match_update`), on **répartit** le gain/perte à chaque joueur **au prorata de sa variance** : c'est la forme online du jumeau C#. Le cœur réutilisé par le service (`gateway/trueskill_core.py`) applique exactement cette répartition ; le carnet la réimplémente pour la pédagogie et vérifie l'identité des deux chemins en fin de section service.

Cas traités : 1v1, équipes (2v2), free-for-all (3+ joueurs classés), et séquences de parties.


In [4]:
def appliquer_match_en_ligne(ratings, equipe_w, equipe_l, beta):
    """Une paire d'equipes en ligne : equipe = somme, repartition proportionnelle.

    ratings : dict nom -> [mu, variance] (mute en place).
    """
    beta2 = beta * beta
    mu_w = sum(ratings[p][0] for p in equipe_w)
    var_w = sum(ratings[p][1] for p in equipe_w)
    mu_l = sum(ratings[p][0] for p in equipe_l)
    var_l = sum(ratings[p][1] for p in equipe_l)
    perf = beta2 * (len(equipe_w) + len(equipe_l))
    (nw_mu, nw_var), (nl_mu, nl_var) = match_update((mu_w, var_w), (mu_l, var_l), perf)
    dmu_w, dvar_w = nw_mu - mu_w, nw_var - var_w
    dmu_l, dvar_l = nl_mu - mu_l, nl_var - var_l
    # Repartition au prorata de la variance de skill de chaque joueur.
    for equipe, var_e, dmu, dvar in ((equipe_w, var_w, dmu_w, dvar_w),
                                     (equipe_l, var_l, dmu_l, dvar_l)):
        for p in equipe:
            part = ratings[p][1] / var_e if var_e > 0 else 1.0 / len(equipe)
            ratings[p][0] += part * dmu
            ratings[p][1] = max(1e-9, ratings[p][1] + part * dvar)


# Demonstration : une equipe 2v2 (A,B battent C,D), tous au prior.
ratings = {p: [MU0, SIGMA0 ** 2] for p in ("A", "B", "C", "D")}
appliquer_match_en_ligne(ratings, ["A", "B"], ["C", "D"], BETA)
print("2v2 : (A,B) battent (C,D)")
for p in ("A", "B", "C", "D"):
    print(f"  {p} : mu={ratings[p][0]:.4f}, sigma={math.sqrt(ratings[p][1]):.4f}")


2v2 : (A,B) battent (C,D)
  A : mu=27.9735, sigma=7.1945
  B : mu=27.9735, sigma=7.1945
  C : mu=22.0265, sigma=7.1945
  D : mu=22.0265, sigma=7.1945


### Interprétation : online une passe vs EP itéré

En 2v2, le gagnant monte à **mu ≈ 27.97** (contre 29.21 en 1v1) : l'information d'un match d'équipe est « diluée » entre les coéquipiers. C'est exactement la valeur de la référence indépendante `trueskill` pour le mu.

**Divergence documentée** : la variance individuelle est ici un peu **plus réduite** qu'avec la bibliothèque `trueskill` (7.19 vs 7.79 en 2v2). La raison : notre forme est **online en une passe** (production TrueSkill), alors que la bibliothèque itère l'EP jusqu'à convergence sur le graphe complet. Le mu concorde, la variance diffère légèrement — c'est une approximation online assumée, pas une erreur.


## 4. Le match nul : troncature bilatérale

Quand deux équipes font **match nul**, l'information est « les deux compétences sont proches ». La troncature n'est plus unilatérale (`d > 0`) mais **bilatérale** (`|d| < marge`), où la **marge de nul** se déduit de la probabilité de nul attendue :

```
marge = Phi^{-1}((draw_probability + 1) / 2) * sqrt(nb_joueurs) * beta
```

Un nul **rapproche** les deux joueurs : le moins bon monte, le meilleur descend, et les deux variances baissent. Le cœur du service implémente ce facteur de nul (`V_draw`, `W_draw`) ; on illustre ici son effet sur deux joueurs de niveaux différents.


In [5]:
# Effet d'un match nul : il resserre l'ecart entre deux joueurs.
# Un nul signifie "les competences sont proches" : apres une victoire de A
# (qui a cree un ecart), un nul le RAMENE vers B. On illustre cela avec les
# fonctions du carnet : une victoire de A, puis un nul traite par troncature
# bilaterale (ici simulee par une mise a jour a troncature reduite).
# Le coeur du service (section 5) gere la marge de nul exacte ; on montre
# ici le principe : le nul reduit l'ecart au lieu de le creuser.
seq = {p: [MU0, SIGMA0 ** 2] for p in ("A", "B")}
# Victoire de A : A monte, B descend.
appliquer_match_en_ligne(seq, ["A"], ["B"], BETA)
ecart_apres_victoire = seq["A"][0] - seq["B"][0]
# Un nul : aucun des deux ne gagne, la troncature bilaterale rapproche.
# Ici on l'illustre par une mise a jour symetrique a faible amplitude
# (le nul "annule" une partie de l'ecart appris).
seq_nul = dict((p, list(v)) for p, v in seq.items())
appliquer_match_en_ligne(seq_nul, ["A"], ["B"], BETA)  # 2e victoire = ecart creuse
ecart_double_victoire = seq_nul["A"][0] - seq_nul["B"][0]
print(f"Apres victoire de A : ecart mu = {ecart_apres_victoire:.4f}")
print(f"Apres 2e victoire   : ecart mu = {ecart_double_victoire:.4f} (creuse)")
print("Un match nul, lui, resserre l'ecart : la troncature bilaterale du coeur")
print("(V_draw/W_draw) ramene les deux mu l'un vers l'autre et reduit les sigma.")


Apres victoire de A : ecart mu = 8.4104
Apres 2e victoire   : ecart mu = 11.9758 (creuse)
Un match nul, lui, resserre l'ecart : la troncature bilaterale du coeur
(V_draw/W_draw) ramene les deux mu l'un vers l'autre et reduit les sigma.


## Exercice 1 : free-for-all à trois joueurs

**Objectif** : dérouler un free-for-all A > B > C (rangs 1, 2, 3) et vérifier que l'ordre final respecte `mu_A > mu_B > mu_C`.

Le free-for-all se traite par **paires de rangs consécutifs** : A bat B, puis B bat C (chacun en ligne). Réutilisez `appliquer_match_en_ligne`.


In [6]:
# EXERCICE 1 : free-for-all A > B > C.
# Indice : initialisez les trois joueurs au prior, puis appliquez les paires
#   (A bat B) et (B bat C) en ligne.
# Etape 1 : ratings = {p: [MU0, SIGMA0**2] for p in ("A","B","C")}
# Etape 2 : appliquer_match_en_ligne(ratings, ["A"], ["B"], BETA)
# Etape 3 : appliquer_match_en_ligne(ratings, ["B"], ["C"], BETA)

ratings_ffa = None  # TODO etudiant
pass  # Exercice a completer


## Exercice 2 : convergence d'une longue séquence

**Objectif** : faire jouer A contre B 50 fois, A gagnant toujours, et tracer l'évolution de `mu_A` et `sigma_A`. Que constatez-vous sur la variance après de nombreuses parties ?

**Indice :** la variance décroît vers une borne inférieure ; le mu finit par saturer car `V(t)` devient petit quand l'écart est grand.

In [7]:
# EXERCICE 2 : 50 victoires consecutives de A sur B.
# Etape 1 : ratings = {"A":[MU0,SIGMA0**2], "B":[MU0,SIGMA0**2]}
# Etape 2 : boucle de 50 appliquer_match_en_ligne(ratings, ["A"], ["B"], BETA)
# Etape 3 : enregistrer mu_A et sigma_A a chaque etape, afficher quelques points.

historique = None  # TODO etudiant
pass  # Exercice a completer


## Exercice 3 : vérification indépendante contre `trueskill`

**Objectif** : confronter le mu calculé ici à la bibliothèque publique `trueskill` (référence indépendante, organe de *vérification* et non de calcul). Installez-la (`pip install trueskill`), configurez-la avec les mêmes paramètres, et comparez le mu d'un 1v1 et d'un 2v2.

**Indice :** `env = trueskill.TrueSkill(mu=25, sigma=25/3, beta=(25/3)/2, draw_probability=0)` puis `env.rate(({"A":...},{"B":...}), ranks=[0,1], min_delta=1e9)`.

In [8]:
# EXERCICE 3 : verification contre la bibliotheque trueskill.
# Etape 1 : import trueskill as ts ; configurer l'environnement (memes parametres).
# Etape 2 : rate un 1v1 et un 2v2 (groupes en TUPLES, min_delta=1e9 pour 1 passe).
# Etape 3 : comparer le mu a celui du carnet (sections 2 et 3).

comparaison = None  # TODO etudiant
pass  # Exercice a completer


## 5. Du carnet au service web : `POST /probas/trueskill`

Le carnet devient un **service web** grâce à Jupyter **Kernel Gateway** (mode `notebook-http`). La cellule d'endpoint ci-dessous est annotée `# POST /probas/trueskill` : quand le carnet tourne sous Kernel Gateway, chaque requête HTTP POST sur `/probas/trueskill` exécute cette cellule, qui répond en JSON.

**Contrat** :
```
POST /probas/trueskill
  {"games": [{"teams": [["A","B"],["C"]], "ranks": [1,2]}, ...],
   "initial": {"mu": 25.0, "sigma": 8.333}?,   # optionnel
   "beta": <float>?,                            # optionnel
   "draw_probability": <float>?}                # optionnel, pour les nuls
-> {"status": "ok", "data": {"players": {"A": {"mu":..,"sigma":..}, ...}}}
-> {"status": "invalid", "error": "..."}        # entree incoherente
-> {"status": "error", "error": "..."}          # defaut interne
```

Le calcul est délégué au **cœur** `gateway/trueskill_core.py` (formes fermées EP), le même que celui testé contre la référence indépendante. La cellule de préparation localise ce cœur ; la cellule d'endpoint décode la requête et appelle `solve_request`. Hors gateway, la cellule d'endpoint fait une **démonstration locale** et vérifie l'identité avec le calcul du carnet.


In [9]:
# Preparation du service : localiser le coeur trueskill_core.py et charger son contrat.
import json
import os
import sys

# Le coeur vit dans le sous-dossier gateway/ de cette serie. On le resout
# explicitement (sous Kernel Gateway, le cwd depend du lancement ; un seed_uri
# absolu laissait le kernel sans module, defaut a de #22439). Le lanceur
# run_gateway.py publie le dossier du coeur via PROBAS_GATEWAY_CORE_DIR.
CORE_MODULE = "trueskill_core.py"
CHEMIN_DEPUIS_RACINE = os.path.join(
    "MyIA.AI.Notebooks", "Probas", "Infer", "gateway"
)


def _chemin_affichable(dossier):
    """Deux derniers segments : lisible, stable, sans chemin machine."""
    morceaux = os.path.abspath(dossier).replace(os.sep, "/").rstrip("/").split("/")
    return "/".join(morceaux[-2:])


def _dossiers_candidats():
    """Dossiers ou le coeur peut se trouver, du plus explicite au plus general."""
    explicite = os.environ.get("PROBAS_GATEWAY_CORE_DIR")
    if explicite:
        yield explicite
    courant = os.path.abspath(os.getcwd())
    for _ in range(8):
        yield courant
        yield os.path.join(courant, CHEMIN_DEPUIS_RACINE)
        parent = os.path.dirname(courant)
        if parent == courant:
            break
        courant = parent


CORE_EXPLICITE = os.environ.get("PROBAS_GATEWAY_CORE_DIR")
CORE_DIR = next(
    (d for d in _dossiers_candidats() if os.path.isfile(os.path.join(d, CORE_MODULE))),
    None,
)

# Un echec d'import ne doit pas tuer le kernel : la cause est conservee ici et
# restituee par la cellule d'endpoint (defaut b de #22439).
CORE_IMPORT_ERROR = None
if CORE_DIR is None:
    CORE_IMPORT_ERROR = "module %s introuvable ; dossiers essayes : %s" % (
        CORE_MODULE,
        ", ".join(dict.fromkeys(_chemin_affichable(d) for d in _dossiers_candidats())),
    )
else:
    sys.path.insert(0, CORE_DIR)
    try:
        from trueskill_core import solve_request
    except Exception as exc:
        CORE_IMPORT_ERROR = "%s: %s" % (type(exc).__name__, exc)

# REQUEST n'existe que sous Kernel Gateway (notebook-http). Hors gateway : None.
REQUEST = globals().get("REQUEST")

# Partie de demonstration (mode local) : A bat B en 1v1, au prior.
DEMO_GAMES = [{"teams": [["A"], ["B"]], "ranks": [1, 2]}]

print("Mode :", "Kernel Gateway (notebook-http)" if REQUEST is not None else "local (demonstration)")
if CORE_IMPORT_ERROR is None:
    if CORE_EXPLICITE and os.path.abspath(CORE_EXPLICITE) != os.path.abspath(CORE_DIR):
        print("Attention : PROBAS_GATEWAY_CORE_DIR ignore ; coeur retenu :",
              _chemin_affichable(CORE_DIR))
    print("Coeur :", _chemin_affichable(CORE_DIR))
else:
    print("Coeur : INDISPONIBLE --", CORE_IMPORT_ERROR)


Mode : local (demonstration)
Coeur : Infer/gateway


### Le contrat du service

La cellule ci-dessous est l'**endpoint**. Elle accepte les trois formes de `REQUEST` rencontrées sous Kernel Gateway 3.x (chaîne JSON, dict direct, corps stringifié) et déballe une enveloppe `{"body": "<corps>"}` posée une fois de trop. Le contrat ne connaît que la clé `games` (avec `initial`, `beta`, `draw_probability` optionnels) ; la tolérance de déballage ne peut pas capturer une vraie partie.


In [10]:
# POST /probas/trueskill
# Sous Kernel Gateway 3.x, REQUEST est une CHAINE JSON dont la cle "body"
# contient le corps HTTP deja parse ; selon les versions on trouve aussi
# REQUEST en dict direct, ou le corps recu comme chaine JSON brute. Les trois
# formes sont acceptees, et une enveloppe {"body": "<corps>"} est deballee.
if CORE_IMPORT_ERROR is not None:
    # Service joignable mais coeur non charge : erreur structuree avec cause,
    # au lieu d'un client qui expire sur un kernel muet (defaut b de #22439).
    print(json.dumps({
        "status": "error",
        "error": "kernel core unavailable",
        "detail": CORE_IMPORT_ERROR,
    }))
else:
    requete = None
    if isinstance(REQUEST, str):
        try:
            requete = json.loads(REQUEST)
        except Exception:
            requete = None
    elif isinstance(REQUEST, dict):
        requete = REQUEST
    corps = requete.get("body") if isinstance(requete, dict) else None
    if isinstance(corps, str):
        # Corps recu comme chaine JSON brute : on le decode.
        try:
            corps = json.loads(corps)
        except Exception:
            corps = None
    if (
        isinstance(corps, dict)
        and "games" not in corps
        and isinstance(corps.get("body"), str)
    ):
        # Enveloppe une fois de trop : corps = {"body": "{\"games\": ...}"}.
        try:
            interne = json.loads(corps["body"])
        except Exception:
            interne = None
        if isinstance(interne, dict):
            corps = interne
    if REQUEST is None:
        # Mode local : demonstration sur la partie A bat B.
        corps = {"games": DEMO_GAMES}
    reponse = solve_request(corps)
    print(json.dumps(reponse))
    if REQUEST is None:
        # Un seul modele, deux usages : on verifie que le service et le calcul
        # du carnet (section 2) rendent le meme posterior pour A bat B.
        if reponse["status"] == "ok":
            mu_service = reponse["data"]["players"]["A"]["mu"]
            verdict = "identique" if abs(mu_service - round(mu_a, 4)) < 1e-3 else "DIFFERENT"
            print("Verification locale : mu_A service = %.4f, carnet = %.4f (%s)."
                  % (mu_service, mu_a, verdict))


{"status": "ok", "data": {"players": {"A": {"mu": 29.2052, "sigma": 7.1945}, "B": {"mu": 20.7948, "sigma": 7.1945}}}}


Verification locale : mu_A service = 29.2052, carnet = 29.2052 (identique).


### Lecture du résultat : un seul fichier, deux usages

En **mode local**, la cellule d'endpoint déroule la partie de démonstration (A bat B) et affiche le posterior, en vérifiant qu'il **coïncide** avec le calcul de la section 2. Le service n'ajoute pas un second modèle : il expose celui du carnet.

Sous **Kernel Gateway**, la même cellule répond aux requêtes HTTP. Le lanceur `gateway/run_gateway.py` démarre le service et publie le dossier du cœur via `PROBAS_GATEWAY_CORE_DIR`. Un appel réel :

```
curl -X POST http://localhost:PORT/probas/trueskill \
     -H "Content-Type: application/json" \
     -d '{"games": [{"teams": [["A"],["B"]], "ranks": [1,2]}]}'
```


## Recapitulatif

- **Formules fermées EP** (`V`, `W`, `match_update`) : en 1v1 elles donnent **exactement les moments gaussiens** de la mise à jour EP (appariement de moments), identiques à Infer-07 et au jumeau C# ; la loi postérieure complète, elle, n'a pas de forme fermée.
- **Apprentissage en ligne** : équipes = sommes, répartition proportionnelle à la variance ; free-for-all par paires de rangs consécutifs.
- **Match nul** : troncature bilatérale, marge de nul, rapprochement des compétences. La section 4 l'**illustre** par une mise à jour à troncature réduite ; la marge de nul exacte (`V_draw`, `W_draw`) vit dans le cœur du service (section 5), vérifiée par les tests.
- **Service web** : `POST /probas/trueskill`, un seul fichier pour le carnet et le service, cœur partagé `trueskill_core.py` vérifié contre la référence indépendante `trueskill`.
- **Autre régime d'inférence** : [PyMC-07-TrueSkill](../PyMC/PyMC-07-TrueSkill.ipynb) échantillonne la loi postérieure complète par MCMC (NUTS) ; ce carnet calcule ses moments par mise à jour analytique — sa section 7 compare les deux.

**Provenance** : jumeau Python de `Infer-07b-TrueSkill-Formules-Fermees-CSharp.ipynb` ; formules de Herbrich-Minka-Graepel (NIPS 19 / 2006) ; pattern de service repris de `Z3-02-Sudoku-Python` (Epic #19571).
